# 03 — Null ve Eksik Veri Analizi

Sentetik veride bilinçli null'lar var — EDA'da **neden eksik?** sorusunu cevaplayacağız.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name != 'eda':
    ROOT = ROOT / 'eda'
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_all
from utils.plots import setup_style

setup_style()
dfs = load_all()

In [ ]:
# Genel null matrisi
null_rows = []
for name, df in dfs.items():
    for col in df.columns:
        n = df[col].isna().sum()
        if n:
            null_rows.append({'tablo': name, 'kolon': col, 'null_sayisi': n,
                              'oran_pct': round(100*n/len(df), 3)})
null_df = pd.DataFrame(null_rows).sort_values('null_sayisi', ascending=False)
display(null_df)

In [ ]:
# Heatmap — tablo × kolon null oranı
pivot = null_df.pivot_table(index='tablo', columns='kolon', values='oran_pct', fill_value=0)
plt.figure(figsize=(14, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Null oranı (%) — tablo × kolon')
plt.tight_layout()
plt.show()

In [ ]:
# transactions.tank_no null — istasyon kırılımı
tx = dfs['transactions']
unmapped = tx[tx['tank_no'].isna()]
print('Unmapped satış:', len(unmapped), f'({100*len(unmapped)/len(tx):.2f}%)')
print(unmapped.groupby('istasyon_kodu').size().sort_values(ascending=False))

In [ ]:
# sicaklik null — tank×gün kümesi mi?
ue1t = dfs['ue1t_30min']
inv = dfs['inventory_30min']
for name, df, col in [('ue1t','ue1t_30min','sicaklik'), ('inventory','inventory_30min','sicaklik')]:
    d = dfs[name.replace('inventory','inventory_30min') if name=='inventory' else 'ue1t_30min']
    sn = d[d['sicaklik'].isna()].copy()
    if len(sn):
        sn['gun'] = sn['saat_1' if 'saat_1' in sn.columns else 'envanter_tarihi'].dt.date
        print(f'\n{name} sicaklik null: {len(sn)} satır')
        print('Tank-gün kümesi (ilk 10):')
        print(sn.groupby(['istasyon_kodu','tank_no','gun']).size().head(10))

In [ ]:
# merkeze_gelis gecikmesi — router arızası imzası
inv = dfs['inventory_30min'].copy()
inv['gecikme_dk'] = (inv['merkeze_gelis_tarihi'] - inv['envanter_tarihi']).dt.total_seconds() / 60
print('Gecikme istatistikleri (dk):')
print(inv['gecikme_dk'].describe())

# En yüksek gecikmeli istasyon-gün
inv['tarih'] = inv['envanter_tarihi'].dt.normalize()
lag_day = inv.groupby(['istasyon_kodu','tarih'])['gecikme_dk'].median().reset_index()
top = lag_day.nlargest(5, 'gecikme_dk')
print('\nEn yüksek medyan gecikme (istasyon-gün):')
display(top)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
inv.boxplot(column='gecikme_dk', by='istasyon_kodu', ax=ax)
ax.set_title('Merkeze geliş gecikmesi (dk) — istasyon')
ax.set_xlabel('İstasyon')
plt.suptitle('')
plt.show()

In [ ]:
# Eksik 30dk satırları — beklenen vs gerçek
tanks = dfs['tanks']
ue1t = dfs['ue1t_30min']
n_gun = dfs['daily'].tarih.nunique()
beklenen = n_gun * 48
cnt = ue1t.groupby(['istasyon_kodu','tank_no']).size().reset_index(name='satir')
tank_list = tanks[['istasyon_kodu','tank_no']]
cnt = tank_list.merge(cnt, on=['istasyon_kodu','tank_no'], how='left').fillna(0)
cnt['eksik'] = beklenen - cnt['satir']
eksik = cnt[cnt['eksik'] > 0]
print(f'Beklenen dönem/tank: {beklenen}')
print(f'Eksik satırı olan tank sayısı: {len(eksik)}')
if len(eksik):
    display(eksik.sort_values('eksik', ascending=False).head(10))

## Doldurma stratejisi notları (feature engineering için)

| Kolon | Öneri |
|---|---|
| `sicaklik` null | Aynı tank-gün interpolasyon veya forward fill |
| `tank_no` null | mapping join veya ayrı "unmapped" grubu |
| `birim_fiyat` null | tutar/litre veya gün medyanı |
| `merkeze_gelis_tarihi` null | Gecikme feature olarak bırak |

Sonraki: tek tank derinlemesine analiz (`04_tek_tank_derinlemesine.ipynb`).